In [1]:
# PReview Data on a map
import geopandas as gpd
from sqlalchemy import create_engine, text
# Reduce Data and Count initial Rows
engine = create_engine('postgresql://markfinlay:@localhost:5432/osm')

# Select all rows from trail_segments
query = "SELECT * FROM trail_segments;"

# Load the data into a GeoDataFrame.
trail_segments_gpdf= gpd.read_postgis(query, engine, geom_col='center_geom').to_crs(epsg=3035)

print(len(trail_segments_gpdf))

# Display the first few rows
trail_segments_gpdf.head()

34313


,id,filename,osm_id,surface_group,center_geom,image_data
0,1,439756915_segment_1,439756915,asphalt,POINT (3464344.127 3230519.951),None
1,2,439756915_segment_2,439756915,asphalt,POINT (3464430.576 3230760.129),None
2,3,439756915_segment_3,439756915,asphalt,POINT (3464261.534 3230889.541),None
3,4,439756915_segment_4,439756915,asphalt,POINT (3464007.959 3230870.7),None
4,5,681268621_segment_1,681268621,asphalt,POINT (3688064.792 3276806.644),None


In [2]:
import io
from PIL import Image
from sqlalchemy import text
from sentinelhub import SentinelHubRequest, DataCollection, MimeType, CRS, BBox, SHConfig
import numpy as np
import geopandas as gpd

# Sentinel Hub configuration (assumed already set up)
config = SHConfig()
config.sh_client_id = 'f3101a77-1835-4be3-ae45-7e9f7c5ff738'
config.sh_client_secret = 'qTPPEOHdlEQdUUDcFZaYGFBpgYii1CWI'

evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B02", "B03", "B04"],
    output: { bands: 3 }
  };
}
function evaluatePixel(sample) {
  var factor = 8;  // Increase brightness
  return [
    Math.min(sample.B04 * factor, 1.0),
    Math.min(sample.B03 * factor, 1.0),
    Math.min(sample.B02 * factor, 1.0)
  ];
}
"""

# Query rows that do not currently have image_data
query = "SELECT * FROM trail_segments WHERE image_data IS NULL"
trail_segments_missing_images = gpd.read_postgis(query, engine, geom_col='center_geom')

print(f"Found {len(trail_segments_missing_images)} segments missing images.")

# Loop through each segment and import the corresponding Sentinel-2 image
for idx, seg in trail_segments_missing_images.iterrows():
    segment_id = seg['id']           # Unique identifier for the segment
    centroid = seg['center_geom']      # Shapely Point in EPSG:3035

    # Create a 320m x 320m bounding box around the centroid
    half_side = 160  # 320 / 2 in meters
    min_x = centroid.x - half_side
    min_y = centroid.y - half_side
    max_x = centroid.x + half_side
    max_y = centroid.y + half_side
    seg_bbox = BBox(bbox=[min_x, min_y, max_x, max_y], crs=CRS('EPSG:3035'))
    
    # Define the image size: 320m extent at 10m/pixel yields a 32x32 image
    size = (32, 32)
    
    # Build the Sentinel Hub request
    request = SentinelHubRequest(
        data_folder=f'sentinel_data_segments/{segment_id}',
        evalscript=evalscript,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=DataCollection.SENTINEL2_L2A,
                time_interval=('2025-01-01', '2025-03-20'),
                mosaicking_order='leastCC'
            )
        ],
        responses=[SentinelHubRequest.output_response('default', MimeType.TIFF)],
        bbox=seg_bbox,
        size=size,
        config=config
    )
    
    try:
        # Retrieve the Sentinel-2 image data (the returned list contains one image)
        seg_image = request.get_data(save_data=True)[0]
    except Exception as e:
        print(f"Error fetching image for segment id {segment_id}: {e}")
        continue

    # If image is in (bands, height, width) shape, transpose to (height, width, bands)
    if seg_image.ndim == 3 and seg_image.shape[0] == 3:
        seg_image_disp = seg_image.transpose(1, 2, 0)
    else:
        seg_image_disp = seg_image

    # Convert the image to uint8 (scale values from 0 to 255 if necessary)
    if seg_image_disp.dtype != np.uint8:
        seg_image_disp_uint8 = (np.clip(seg_image_disp, 0, 1) * 255).astype(np.uint8)
    else:
        seg_image_disp_uint8 = seg_image_disp

    # Convert the image to a PNG byte stream
    im = Image.fromarray(seg_image_disp_uint8)
    buffer = io.BytesIO()
    im.save(buffer, format="PNG")
    image_bytes = buffer.getvalue()

    # SQL update statement to update the image_data column for the segment row
    update_sql = """
    UPDATE trail_segments
    SET image_data = :image_data
    WHERE id = :id;
    """
    
    metadata = {"id": segment_id, "image_data": image_bytes}
    with engine.begin() as conn:
        conn.execute(text(update_sql), metadata)
    
print("Sentinel-2 image updates complete for all segments missing images.")


Found 34313 segments missing images.
Error fetching image for segment id 1: Failed to download from:
https://services.sentinel-hub.com/api/v1/process
with HTTPError:
403 Client Error: Forbidden for url: https://services.sentinel-hub.com/api/v1/process
Server response: "{"status": 403, "reason": "Forbidden", "message": "Invalid or expired account.", "code": "COMMON_INSUFFICIENT_PERMISSIONS"}"
Error fetching image for segment id 2: Failed to download from:
https://services.sentinel-hub.com/api/v1/process
with HTTPError:
403 Client Error: Forbidden for url: https://services.sentinel-hub.com/api/v1/process
Server response: "{"status": 403, "reason": "Forbidden", "message": "Invalid or expired account.", "code": "COMMON_INSUFFICIENT_PERMISSIONS"}"
Error fetching image for segment id 3: Failed to download from:
https://services.sentinel-hub.com/api/v1/process
with HTTPError:
403 Client Error: Forbidden for url: https://services.sentinel-hub.com/api/v1/process
Server response: "{"status": 403


KeyboardInterrupt



In [ ]:
import pandas as pd
import io
from PIL import Image
import matplotlib.pyplot as plt

# Query rows that have image data
query = "SELECT id, filename, image_data FROM trail_segments WHERE image_data IS NOT NULL"
segments_with_images = pd.read_sql(query, engine)

# Loop through the results and display each image
for idx, row in segments_with_images.iterrows():
    segment_id = row['id']
    filename = row['filename']
    image_bytes = row['image_data']
    
    # Convert byte data to an image using PIL
    im = Image.open(io.BytesIO(image_bytes))
    
    plt.figure(figsize=(3, 3))
    plt.imshow(im)
    plt.title(f"Segment {segment_id} - {filename}")
    plt.axis('off')
    plt.show()
